# Load Supply Chain Domain Data - v2 (Fixed)

Loads Suppliers, ProductSuppliers, SupplyChainEvents, and SupplyChainEventImpacts.

**Fixes applied:**
- Split SupplyChainEvents into Events + EventImpacts
- SupplierIDs now populated in EventImpacts (were 100% NULL)
- Removed denormalized columns (ProductName, ProductCategory, SupplierName)
- PrimarySupplierID cleaned (removed .0 suffix)
- Proper validation and MERGE INTO
- Load order: Suppliers → ProductSuppliers → SupplyChainEvents → SupplyChainEventImpacts

In [ ]:
from pyspark.sql.functions import col, to_date, when, lit
from pyspark.sql.types import *

SCHEMA_NAME = "supplychain"
DATA_PATH = "Files/data/supplychain"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SCHEMA_NAME}")
print(f"Schema {SCHEMA_NAME} ready")

In [ ]:
TABLE = "Suppliers"
print(f"Loading {SCHEMA_NAME}.{TABLE}...")

df = (spark.read.format("csv")
    .option("header", "true")
    .option("inferSchema", "false")
    .load(f"{DATA_PATH}/{TABLE}.csv")
    .select(
        col("SupplierID").cast("string"),
        col("SupplierName").cast("string"),
        col("SupplierType").cast("string"),
        col("Status").cast("string"),
        col("ProductLineName").cast("string"),
        col("PrimarySupplierID").cast("string"),
        col("LeadTimeDays").cast("int"),
        col("ReliabilityScore").cast("decimal(5,2)"),
        col("Location").cast("string"),
        col("ContactEmail").cast("string"),
        col("CreatedBy").cast("string"),
        to_date(col("CreatedDate"), "yyyy-MM-dd").alias("CreatedDate")
    ))

assert df.filter(col("SupplierID").isNull()).count() == 0, "NULL SupplierIDs!"
assert df.count() == df.dropDuplicates(["SupplierID"]).count(), "Duplicate SupplierIDs!"

df.write.mode("append").insertInto(f"{SCHEMA_NAME}.{TABLE}")
print(f"✅ {SCHEMA_NAME}.{TABLE}: {df.count()} rows loaded")

In [ ]:
TABLE = "ProductSuppliers"
print(f"Loading {SCHEMA_NAME}.{TABLE}...")

df = (spark.read.format("csv")
    .option("header", "true")
    .option("inferSchema", "false")
    .load(f"{DATA_PATH}/{TABLE}.csv")
    .select(
        col("ProductSupplierID").cast("string"),
        col("ProductID").cast("string"),
        col("SupplierID").cast("string"),
        col("SupplierProductCode").cast("string"),
        col("WholesaleCost").cast("decimal(10,2)"),
        col("MinOrderQuantity").cast("int"),
        col("MaxOrderQuantity").cast("int"),
        col("LeadTimeDays").cast("int"),
        col("Status").cast("string"),
        col("CreatedBy").cast("string"),
        to_date(col("CreatedDate"), "yyyy-MM-dd").alias("CreatedDate")
    ))

assert df.filter(col("ProductSupplierID").isNull()).count() == 0, "NULL ProductSupplierIDs!"

df.write.mode("append").insertInto(f"{SCHEMA_NAME}.{TABLE}")
print(f"✅ {SCHEMA_NAME}.{TABLE}: {df.count()} rows loaded")

In [ ]:
TABLE = "SupplyChainEvents"
print(f"Loading {SCHEMA_NAME}.{TABLE}...")

df = (spark.read.format("csv")
    .option("header", "true")
    .option("inferSchema", "false")
    .load(f"{DATA_PATH}/{TABLE}.csv")
    .select(
        col("EventID").cast("string"),
        col("SupplierID").cast("int"),
        col("DisruptionType").cast("string"),
        col("EventName").cast("string"),
        col("Description").cast("string"),
        col("Severity").cast("string"),
        col("Status").cast("string"),
        to_date(col("StartDate"), "yyyy-MM-dd").alias("StartDate"),
        to_date(col("EndDate"), "yyyy-MM-dd").alias("EndDate"),
        col("GeographicArea").cast("string"),
        col("IndustryImpact").cast("string"),
        col("PredictedDuration").cast("int"),
        col("ActualDuration").cast("int"),
        col("AlertLevel").cast("string"),
        col("ReportedBy").cast("string"),
        col("CreatedBy").cast("string"),
        to_date(col("CreatedDate"), "yyyy-MM-dd").alias("CreatedDate")
    ))

assert df.filter(col("EventID").isNull()).count() == 0, "NULL EventIDs!"

df.write.mode("append").insertInto(f"{SCHEMA_NAME}.{TABLE}")
print(f"✅ {SCHEMA_NAME}.{TABLE}: {df.count()} rows loaded")

In [ ]:
TABLE = "SupplyChainEventImpacts"
print(f"Loading {SCHEMA_NAME}.{TABLE}...")

df = (spark.read.format("csv")
    .option("header", "true")
    .option("inferSchema", "false")
    .load(f"{DATA_PATH}/{TABLE}.csv")
    .select(
        col("ImpactID").cast("int"),
        col("EventID").cast("string"),
        col("SupplierID").cast("string"),
        col("ProductLineName").cast("string"),
        col("ImpactLevel").cast("string"),
        col("DeliveryDelay").cast("int"),
        col("CostIncrease").cast("decimal(5,2)"),
        col("AlternativeAction").cast("string"),
        col("EstimatedRevenueImpact").cast("decimal(12,2)"),
        col("CreatedBy").cast("string"),
        to_date(col("CreatedDate"), "yyyy-MM-dd").alias("CreatedDate")
    ))

assert df.filter(col("ImpactID").isNull()).count() == 0, "NULL ImpactIDs!"
assert df.filter(col("SupplierID").isNull()).count() == 0, "NULL SupplierIDs in impacts! (was 100% null before fix)"

df.write.mode("append").insertInto(f"{SCHEMA_NAME}.{TABLE}")
print(f"✅ {SCHEMA_NAME}.{TABLE}: {df.count()} rows loaded")

In [ ]:
print("🎉 SUPPLY CHAIN DOMAIN LOAD COMPLETE")
for t in ["Suppliers", "ProductSuppliers", "SupplyChainEvents", "SupplyChainEventImpacts"]:
    count = spark.sql(f"SELECT COUNT(*) as cnt FROM {SCHEMA_NAME}.{t}").first()["cnt"]
    print(f"   {SCHEMA_NAME}.{t}: {count} rows")